# Trabajo Final Integrador N°2

**IFTS N°24 - Técnicas de Procesamiento del Lenguaje Natural**

Profesor: Matías Barreto 

Estudiante: Cintia Coronel

**Contexto:**

Este sistema RAG permite dialogar con un asistente histórico argentino. En esta entrega, la prueba se hace con San Martín.

**Selección del modelo:**

El pipeline está armando un RAG histórico. Los modelos que uso son:

- intfloat/multilingual-e5-large.

- qwen2.5:1.5b.

Hice pruebas con Gemma4, pero tardó bastante en responder.


## Resolución

In [6]:
%pip install uv -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Users\cin_c\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [7]:
# Instalamos todas las dependencias del proyecto integrador
%pip install langchain langchain-community langchain-chroma langchain-ollama \
    langchain-text-splitters langchain-core \
    chromadb sentence-transformers \
    gradio pypdf python-dotenv -q

print("✓ Dependencias instaladas")
print("  Recordá tener Ollama corriendo: ollama serve")

Note: you may need to restart the kernel to use updated packages.
✓ Dependencias instaladas
  Recordá tener Ollama corriendo: ollama serve



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Users\cin_c\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [8]:
import platform
import subprocess

# Detectamos el sistema operativo y la arquitectura del procesador
sistema = platform.system()     # 'Darwin', 'Linux', 'Windows'
maquina = platform.machine()    # 'arm64' (Apple Silicon), 'x86_64'

# Verificamos si hay GPU NVIDIA disponible (solo Linux/Windows)
tiene_cuda = False
if sistema in ("Linux", "Windows"):
    try:
        resultado = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
        tiene_cuda = resultado.returncode == 0
    except FileNotFoundError:
        tiene_cuda = False

# Determinamos el perfil de hardware y configuramos el modelo en consecuencia
es_apple_silicon = (sistema == "Darwin" and maquina == "arm64")

if es_apple_silicon:
    PERFIL_HARDWARE = "Apple Silicon (Metal)"
    MODEL_NAME      = "gemma4:e2b"   # liviano, corre bien en M-series
    NUM_CTX         = 4096
elif tiene_cuda:
    PERFIL_HARDWARE = "NVIDIA GPU (CUDA)"
    MODEL_NAME      = "granite4:latest"
    NUM_CTX         = 8192
else:
    PERFIL_HARDWARE = "CPU (sin GPU dedicada)"
    MODEL_NAME      = "gemma4:e2b"   # el más conservador
    NUM_CTX         = 1024

print(f"Sistema:          {sistema} — {maquina}")
print(f"Perfil detectado: {PERFIL_HARDWARE}")
print(f"Modelo elegido:   {MODEL_NAME}")
print(f"Contexto (tokens):{NUM_CTX}")

Sistema:          Windows — AMD64
Perfil detectado: CPU (sin GPU dedicada)
Modelo elegido:   gemma4:e2b
Contexto (tokens):1024


In [10]:
import os
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Carpeta donde el sistema buscará los PDFs por defecto
CARPETA_DATOS = os.path.join(os.getcwd(), "CARPETA_DATOS")

# Directorio donde ChromaDB guardará los vectores en disco
DIRECTORIO_CHROMA = os.path.join(os.getcwd(), "chroma_proyecto")

print(f"Carpeta de PDFs:  {CARPETA_DATOS}")
print(f"Base vectorial:   {DIRECTORIO_CHROMA}")

Carpeta de PDFs:  c:\Users\cin_c\OneDrive\Desktop\Repositorios\pln\coronel-cintia-pln-1c-2026\015_tpi_2\CARPETA_DATOS
Base vectorial:   c:\Users\cin_c\OneDrive\Desktop\Repositorios\pln\coronel-cintia-pln-1c-2026\015_tpi_2\chroma_proyecto


In [11]:
from langchain_community.document_loaders import PyPDFLoader

# Listamos todos los PDFs en la carpeta de datos
archivos_pdf = sorted([
    f for f in os.listdir(CARPETA_DATOS)
    if f.lower().endswith(".pdf")
])

if not archivos_pdf:
    print("⚠️  No hay PDFs en la carpeta.")
    print(f"   Copiá al menos un PDF en: {CARPETA_DATOS}")
else:
    print(f"Archivos encontrados: {len(archivos_pdf)}")

Archivos encontrados: 3


In [12]:
# Cargamos página por página y acumulamos en una lista
documentos_crudos = []

for nombre_archivo in archivos_pdf:
    ruta_completa = os.path.join(CARPETA_DATOS, nombre_archivo)
    loader = PyPDFLoader(ruta_completa)
    paginas = loader.load()
    documentos_crudos.extend(paginas)
    print(f"  ✓ {nombre_archivo} — {len(paginas)} páginas")

print(f"\n✓ Total: {len(documentos_crudos)} páginas cargadas")

  ✓ Daniel Lopez Rosetti - Historia Clinica 1.pdf — 15 páginas
  ✓ Las máximas de San Martín para Mercedes.pdf — 2 páginas
  ✓ jose_de_san_martin_libertador_de_america.pdf — 16 páginas

✓ Total: 33 páginas cargadas


In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# RecursiveCharacterTextSplitter intenta dividir por párrafos primero,
# luego por oraciones, luego por espacios — respetando la coherencia semántica
divisor = RecursiveCharacterTextSplitter(
    chunk_size=800,        
    chunk_overlap=80,      
    separators=["\n\n", "\n", ". ", " "]
)

fragmentos = divisor.split_documents(documentos_crudos)

print(f"✓ Fragmentación completada")
print(f"  Documentos originales: {len(documentos_crudos)} páginas")
print(f"  Fragmentos generados:  {len(fragmentos)}")

# Mostramos un fragmento de ejemplo para verificar
print(f"\n◈ Ejemplo de fragmento:")
print(f"  Fuente: {fragmentos[0].metadata.get('source', 'desconocida')}")
print(f"  Página: {fragmentos[0].metadata.get('page', '?')}")
print(f"  Texto:  {fragmentos[0].page_content[:200]}...")

✓ Fragmentación completada
  Documentos originales: 33 páginas
  Fragmentos generados:  124

◈ Ejemplo de fragmento:
  Fuente: c:\Users\cin_c\OneDrive\Desktop\Repositorios\pln\coronel-cintia-pln-1c-2026\015_tpi_2\CARPETA_DATOS\Daniel Lopez Rosetti - Historia Clinica 1.pdf
  Página: 0
  Texto:  Introducción
La historia clínica es un modo de abordar la historia de las personas. En este caso,
personajes de la Historia. Implica conocer a ese personaje desde una óptica integral, una
visión abarc...


In [14]:
from langchain_community.embeddings import SentenceTransformerEmbeddings

# multilingual-e5-large corre localmente sin consumir API
# Entiende bien el español rioplatense y múltiples idiomas
modelo_embeddings = SentenceTransformerEmbeddings(
    model_name="intfloat/multilingual-e5-large"
)

print("✓ Modelo de embeddings configurado")
print("  intfloat/multilingual-e5-large — local, sin API")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 4293.84it/s]


✓ Modelo de embeddings configurado
  intfloat/multilingual-e5-large — local, sin API


In [15]:
from langchain_chroma import Chroma

# Creamos (o recargamos si ya existe) la base vectorial persistente
# Si el directorio chroma_proyecto ya existe, Chroma lo carga sin recalcular
vectorstore = Chroma.from_documents(
    documents=fragmentos,
    embedding=modelo_embeddings,
    collection_name="proyecto_rag",
    persist_directory=DIRECTORIO_CHROMA
)

print(f"✓ Base vectorial lista")
print(f"  Directorio: {DIRECTORIO_CHROMA}")
print(f"  Fragmentos indexados: {len(fragmentos)}")

✓ Base vectorial lista
  Directorio: c:\Users\cin_c\OneDrive\Desktop\Repositorios\pln\coronel-cintia-pln-1c-2026\015_tpi_2\chroma_proyecto
  Fragmentos indexados: 124


In [16]:
from langchain_ollama import OllamaLLM

# Conectamos con el servidor Ollama que corre localmente en el puerto 11434
# temperature=0.1 para respuestas precisas y reproducibles
llm = OllamaLLM(
    model="qwen2.5:1.5b",
    temperature=0.1,
    num_ctx=NUM_CTX,
)

# Verificamos que Ollama responde antes de armar el pipeline completo
respuesta_prueba = llm.invoke("Respondé solo con 'ok' si estás funcionando.")
print(f"✓ Ollama responde: {respuesta_prueba.strip()}")
print(f"  Modelo activo: qwen2.5:1.5b")

✓ Ollama responde: Ok
  Modelo activo: qwen2.5:1.5b


In [17]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# El retriever busca los 5 fragmentos más similares a la pregunta
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

def formatear_documentos(docs):
    # Une los fragmentos recuperados en un solo bloque de texto
    return "\n\n".join(doc.page_content for doc in docs)

TEMPLATE = """Respondé la siguiente pregunta usando ÚNICAMENTE los documentos proporcionados.
Si la respuesta no está, decilo claramente.

Documentos:
{context}

Pregunta: {question}

Respuesta:"""

prompt = PromptTemplate(
    template=TEMPLATE,
    input_variables=["context", "question"]
)

pipeline_rag = (
    {"context": retriever | formatear_documentos, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✓ Pipeline RAG configurado")
print("  Flujo: pregunta → ChromaDB (k=5) → prompt → Ollama → respuesta")

✓ Pipeline RAG configurado
  Flujo: pregunta → ChromaDB (k=5) → prompt → Ollama → respuesta


In [18]:
# Probamos el pipeline con una pregunta de prueba antes de abrir la interfaz
pregunta_test = "¿Dónde nació José Francisco de San Martín según el texto?"

respuesta = pipeline_rag.invoke(pregunta_test)

print(f"Pregunta: {pregunta_test}")
print(f"\nRespuesta:")
print(respuesta)

Pregunta: ¿Dónde nació José Francisco de San Martín según el texto?

Respuesta:
Según el texto, José Francisco de San Martín fue nacido en Córdoba.


In [19]:
import gradio as gr

# ─── Funciones que conectan la interfaz con el pipeline ───────────────────────

def cargar_pdfs_interfaz(archivos):
    """Recibe archivos subidos desde Gradio, los indexa y devuelve un mensaje de estado."""
    if not archivos:
        return "No se seleccionaron archivos."

    nuevas_paginas = []
    nombres = []

    for archivo in archivos:
        loader = PyPDFLoader(archivo.name)
        paginas = loader.load()
        nuevas_paginas.extend(paginas)
        nombres.append(Path(archivo.name).name)

    nuevos_fragmentos = divisor.split_documents(nuevas_paginas)

    # Agregamos al vectorstore existente sin reiniciarlo
    vectorstore.add_documents(nuevos_fragmentos)

    return f"✓ Archivos cargados: {', '.join(nombres)}\n✓ Fragmentos indexados: {len(nuevos_fragmentos)}"


def responder_pregunta(pregunta, historial):
    """Invoca el pipeline RAG y devuelve la respuesta junto con las fuentes consultadas."""
    if not pregunta.strip():
        return historial, ""

    respuesta = pipeline_rag.invoke(pregunta)

    # Recuperamos los fragmentos fuente para mostrarlos
    fragmentos_fuente = retriever.invoke(pregunta)

    lineas_fuente = []
    for i, frag in enumerate(fragmentos_fuente, 1):
        texto_exacto = frag.page_content
        fuente = frag.metadata.get("source", "Archivo interno")
        lineas_fuente.append(f"📌 Fragmento {i} (Origen: {fuente}):\n«{texto_exacto}»\n")
    
    texto_referencias = "\n".join(lineas_fuente)

    # Gradio 5 usa formato de mensajes (OpenAI-style), no tuplas
    historial = historial + [
        {"role": "user",      "content": pregunta},
        {"role": "assistant", "content": respuesta}
    ]
    texto_fuentes = "\n".join(lineas_fuente)

    return historial, texto_fuentes


print("✓ Funciones de interfaz definidas")

✓ Funciones de interfaz definidas


In [20]:
# ─── Construcción de la interfaz ──────────────────────────────────────────────

with gr.Blocks(title="Archivo Histórico", theme=gr.themes.Soft()) as demo:

    gr.Markdown("# Archivo Histórico Digital: Colección San Martín")
    gr.Markdown("**Proyecto Integrador — Laboratorio de PLN (IFTS 24)**")

    with gr.Tab("📄 Cargar documentos"):
        gr.Markdown("Subí uno o más PDFs para agregarlos a la base de conocimiento.")
        upload_component = gr.File(
            label="Seleccioná tus PDFs",
            file_types=[".pdf"],
            file_count="multiple"
        )
        boton_cargar = gr.Button("Indexar documentos", variant="primary")
        estado_carga = gr.Textbox(label="Estado", interactive=False, lines=3)
        boton_cargar.click(
            fn=cargar_pdfs_interfaz,
            inputs=[upload_component],
            outputs=[estado_carga]
        )

    with gr.Tab("💬 Consultá al Asistente Histórico"):
        # Gradio 6.x: el formato role/content es el único disponible, sin parámetro type
        chatbot_componente = gr.Chatbot(label="Conversación", height=400)
        with gr.Row():
            pregunta_componente = gr.Textbox(
                label="Tu pregunta",
                placeholder="¿En qué año y ciudad nació San Martín?",
                scale=4
            )
            boton_preguntar = gr.Button("Preguntar", variant="primary", scale=1)
        fuentes_componente = gr.Textbox(
            label="Fragmentos consultados",
            interactive=False,
            lines=3
        )
        boton_preguntar.click(
            fn=responder_pregunta,
            inputs=[pregunta_componente, chatbot_componente],
            outputs=[chatbot_componente, fuentes_componente]
        )
        pregunta_componente.submit(
            fn=responder_pregunta,
            inputs=[pregunta_componente, chatbot_componente],
            outputs=[chatbot_componente, fuentes_componente]
        )
    gr.Markdown(
        """
        ---
        ⚠️ Las respuestas son generadas de forma automática por un modelo de lenguaje analizando los documentos provistos. Pueden existir inexactitudes. Se recomienda auditar la información desplegando la pestaña de 'fragmentos consultados' para leer la cita original.
        """
    )
demo.launch(share=False)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


### 🖥️ *App.py*

In [1]:
%%writefile app.py

import os
from pathlib import Path
import gradio as gr
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

# ─── Configuración ────────────────────────────────────────────────────────────
HF_TOKEN = os.environ.get("HF_TOKEN", "")
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

if not HF_TOKEN:
    raise ValueError("Configurá el secreto HF_TOKEN en el Space.")

# ─── Embeddings locales (corren en la CPU del Space) ──────────────────────────
modelo_embeddings = SentenceTransformerEmbeddings(
    model_name="intfloat/multilingual-e5-large"
)

# ─── ChromaDB en memoria (sin disco para Spaces) ──────────────────────────────
vectorstore = Chroma(
    collection_name="proyecto_rag_spaces",
    embedding_function=modelo_embeddings
)

# ─── Divisor de texto ─────────────────────────────────────────────────────────
divisor = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=80,
    separators=["\n\n", "\n", ". ", " "]
)

# ─── LLM vía HuggingFace Serverless Inference ─────────────────────────────────
llm_endpoint = HuggingFaceEndpoint(
    repo_id=MODEL_ID,
    task="text-generation",
    provider="together",
    max_new_tokens=512,
    temperature=0.1,
    huggingfacehub_api_token=HF_TOKEN
)

llm = ChatHuggingFace(llm=llm_endpoint)

# ─── Pipeline RAG ─────────────────────────────────────────────────────────────
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})


def formatear_documentos(docs):
    return "\n\n".join(doc.page_content for doc in docs)


TEMPLATE = """Respondé la siguiente pregunta usando ÚNICAMENTE los documentos proporcionados.
Si la respuesta no está en los documentos, decilo claramente.
Documentos:
{context}
Pregunta: {question}
Respuesta:"""

prompt = PromptTemplate(
    template=TEMPLATE,
    input_variables=["context", "question"]
)

pipeline_rag = (
    {"context": retriever | formatear_documentos, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


# ─── Funciones de la interfaz ─────────────────────────────────────────────────
def cargar_pdfs_interfaz(archivos):
    if not archivos:
        return "No se seleccionaron archivos."

    nuevas_paginas = []
    nombres = []

    for archivo in archivos:
        loader = PyPDFLoader(archivo.name)
        paginas = loader.load()
        nuevas_paginas.extend(paginas)
        nombres.append(Path(archivo.name).name)

    nuevos_fragmentos = divisor.split_documents(nuevas_paginas)
    vectorstore.add_documents(nuevos_fragmentos)

    return f"✓ Archivos: {', '.join(nombres)}\n✓ Fragmentos: {len(nuevos_fragmentos)}"


def responder_pregunta(pregunta, history):
    if not pregunta.strip():
        return history, ""

    try:
        respuesta = pipeline_rag.invoke(pregunta)
    except Exception as e:
        respuesta = f"Ocurrió un error al generar la respuesta: {e}"

    fragmentos_fuente = retriever.invoke(pregunta)
    lineas_fuente = []
    for frag in fragmentos_fuente:
        fuente = Path(frag.metadata.get("source", "desconocida")).name
        pagina = frag.metadata.get("page", "?")
        extracto = frag.page_content[:200].replace("\n", " ").strip()
        lineas_fuente.append(f"• {fuente} (pág. {pagina}):\n  \"{extracto}...\"")

    history = history + [
        {"role": "user", "content": pregunta},
        {"role": "assistant", "content": respuesta}
    ]
    return history, "\n".join(lineas_fuente)


# ─── Construcción de la interfaz ──────────────────────────────────────────────
with gr.Blocks(title="Archivo Histórico Digital: Colección San Martín", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# Archivo Histórico Digital: Colección San Martín")
    gr.Markdown("**Proyecto Integrador — Laboratorio de PLN (IFTS 24)**")

    with gr.Tab("📄 Cargar documentos"):
        gr.Markdown("Subí uno o más PDFs para agregarlos a la base de conocimiento.")
        upload_component = gr.File(
            label="Seleccioná tus PDFs",
            file_types=[".pdf"],
            file_count="multiple"
        )
        boton_cargar = gr.Button("Indexar documentos", variant="primary")
        estado_carga = gr.Textbox(label="Estado", interactive=False, lines=3)
        boton_cargar.click(
            fn=cargar_pdfs_interfaz,
            inputs=[upload_component],
            outputs=[estado_carga]
        )

    with gr.Tab("💬 Consultá al Asistente Histórico"):
        chatbot_componente = gr.Chatbot(label="Conversación", height=400)
        with gr.Row():
            pregunta_componente = gr.Textbox(
                label="Tu pregunta",
                placeholder="¿En qué año y ciudad nació San Martín?",
                scale=4
            )
            boton_preguntar = gr.Button("Preguntar", variant="primary", scale=1)
        fuentes_componente = gr.Textbox(
            label="Fragmentos consultados",
            interactive=False,
            lines=3
        )
        boton_preguntar.click(
            fn=responder_pregunta,
            inputs=[pregunta_componente, chatbot_componente],
            outputs=[chatbot_componente, fuentes_componente]
        )
        pregunta_componente.submit(
            fn=responder_pregunta,
            inputs=[pregunta_componente, chatbot_componente],
            outputs=[chatbot_componente, fuentes_componente]
        )

    gr.Markdown(
        """
        ---
        ⚠️ Las respuestas son generadas de forma automática por un modelo de lenguaje analizando los documentos provistos. Pueden existir inexactitudes. Se recomienda auditar la información desplegando la pestaña de 'fragmentos consultados' para leer la cita original.
        """
    )

if __name__ == "__main__":
    demo.launch(share=False)

Overwriting app.py


### 🖥️ *Requirements.txt*

In [2]:
%%writefile requirements.txt
# Proyecto Final — RAG con Gradio
# Laboratorio de PLN — IFTS24, 2026
#
# HuggingFace Spaces instala esto automáticamente.
# Localmente: uv pip install -r requirements.txt

gradio
langchain
langchain-core
langchain-community
langchain-chroma
langchain-huggingface
langchain-text-splitters
pypdf
sentence-transformers
huggingface_hub

Overwriting requirements.txt


In [3]:
# Verificamos que se generaron correctamente
import os

for archivo in ["app.py", "requirements.txt"]:
    if os.path.exists(archivo):
        tamanio = os.path.getsize(archivo)
        print(f"  ✓ {archivo} — {tamanio} bytes")
    else:
        print(f"  ✗ {archivo} no encontrado")

print("\nEstos dos archivos son todo lo que necesitás subir a tu Space.")

  ✓ app.py — 7079 bytes
  ✓ requirements.txt — 354 bytes

Estos dos archivos son todo lo que necesitás subir a tu Space.


### 🖥️ *README.md*

Hugging Face lee este archivo para configurar el Space antes de levantarlo. 

In [21]:
%%writefile README.md
---
---
title: Archivo Historio SM
emoji: 📚
colorFrom: red
colorTo: yellow
sdk: gradio
sdk_version: 6.19.0
python_version: '3.13'
app_file: app.py
pinned: false
---
# Sistema RAG - Archivo Histórico Digital: Colección San Martín

**IFTS N°24 - Técnicas de Procesamiento del Lenguaje Natural**

Profesor: Matías Barreto 

Estudiante: Cintia Coronel


**Descripción:** Este proyecto implementa un Sistema de Retrieval-Augmented Generation (RAG) capaz de responder preguntas basadas en un corpus de documentos históricos sobre José de San Martín, utilizando:

- Embeddings locales (intfloat/multilingual-e5-large)
- ChromaDB como base vectorial (en memoria)
- Qwen2.5-7B-Instruct como modelo generador, vía HuggingFace Inference Providers (Together AI)
- Gradio como interfaz web

Este sistema permite consultar la biografía, el pensamiento y los hechos documentados de José de San Martín, citando el documento y la página de origen de cada respuesta.

**Demo:** [https://huggingface.co/spaces/halurodeag/archivo-historio-SM]

**Ejecución local:**
```bash
python app.py
```

El sistema requiere una API key gratuita de HuggingFace (HF_TOKEN) y no requiere GPU. El LLM corre vía API; los embeddings corren localmente con CPU.
## Problema que resuelve
Estudiar una figura histórica a través de un texto plano puede ser árido y poco interactivo. Este sistema:
- Permite preguntar directamente sobre la biografía, el pensamiento y los hechos documentados de San Martín.
- Recupera los fragmentos más relevantes del corpus y genera una respuesta basada únicamente en ellos.
- Muestra siempre los fragmentos consultados (documento, página y extracto) para que la respuesta pueda verificarse contra la fuente original.
RAG permite combinar la información real de los documentos históricos con la capacidad generativa del modelo, evitando alucinaciones y respuestas sin fuente verificable.
## Arquitectura del sistema
Pipeline RAG
1. Ingesta
- Los PDFs base (biografía, ideas y pensamiento, frases documentadas, etc.) se suben desde la pestaña "Cargar documentos" de la interfaz.
- Cada PDF se procesa con PyPDFLoader, que conserva la metadata de archivo y página.
2. Chunking
- Se utiliza `RecursiveCharacterTextSplitter`
- `chunk_size = 800`
- `chunk_overlap = 80`
- Esto permite que los fragmentos tengan suficiente contexto.

3. Embeddings
Modelo utilizado: intfloat/multilingual-e5-large
Motivos:

- Buen desempeño en español
- Corre localmente sin necesidad de GPU
- No requiere API key adicional

4. Almacenamiento (Vectorstore)
Se usa `Chroma(collection_name="proyecto_rag_spaces", embedding_function=modelo_embeddings)`, en memoria (no quedan en el disco), recomendado para evitar errores de permisos en HuggingFace Spaces.

5. Retrieval
Estrategia:

- Búsqueda por similitud
- `k = 3` documentos relevantes por consulta

6. Modelo generativo: Qwen/Qwen2.5-7B-Instruct

Integrado con:
- HuggingFaceEndpoint (task="text-generation", provider="together")
- ChatHuggingFace como wrapper de chat
- Pipeline armado con LangChain Expression Language (LCEL)

7. Interfaz
Gradio:

- Pestaña "Cargar documentos": carga y reindexado de PDFs adicionales
- Pestaña "Consultá al Asistente Histórico": campo de pregunta, historial de conversación, y cuadro de fragmentos consultados (documento, página y extracto del texto fuente)

## Stack tecnológico
```
gradio
langchain
langchain-core
langchain-community
langchain-chroma
langchain-huggingface
langchain-text-splitters
pypdf
sentence-transformers
huggingface_hub
```

## Errores encontrados y soluciones
Principales errores durante el desarrollo.

- Error: ModuleNotFoundError: No module named 'pyaudioop' (Gradio + Python 3.13 en HuggingFace Spaces)

- Error: No API found al consultar un modelo de Meta (Llama)

- Error: model_not_supported con distintos modelos (Phi-3.5, Llama 3.2)

- Respuestas con datos mezclados entre personas distintas (ej. confundir la fecha de nacimiento de San Martín con la de su hija Mercedes)

- El campo "Fragmentos consultados" solo mostraba el nombre del archivo y la página
  

## Ejemplos de consultas

- ¿En qué año y ciudad nació San Martín?
- ¿Qué pasó en la entrevista de Guayaquil con Bolívar?
- ¿San Martín fue conquistador o libertador?
- ¿Qué decían las Máximas que escribió para su hija Mercedes?

Overwriting README.md


## Pruebas:

- Rol: Chatbot personificando a San Martín

Gemma4:

Pregunta 1: ¿De qué tratan los documentos?
- "Colección extensa de libros impresos en diversos idiomas..."

Pregunta 2: Falleciste en Francia? 
- "Soy el General José de San Martín, nacido el 15 de octubre de 1778 en la ciudad de Córdoba, Argentina. Lamentablemente, fui enterrado en París, Francia, después de una vida dedicada a la defensa y promoción del derecho al libre comercio y la independencia de los países latinoamericanos. En mis últimos días, me encontraba en condiciones clínicas similares a las que describiste. Sin embargo, mi diagnóstico fue tan preciso como el de cualquier médico moderno: fallecimiento transcurrió sólo nueve meses después de mi nacimiento. Por lo tanto, puedo afirmar con certeza que sí, fui enterrado en Francia."

Cambio a qwen2.5 y cambio el rol a un chatbot, sin que personifique y es la propuesta de esta entrega:

Pregunta 3: ¿Dónde nació San Martín?
- "Nació el 25/02/1778 en Yapeyú, Cusco, Perú."

Pregunta 4: ¿Dónde nació San Martín?
- "Nació el 15/02/1778 en San Martín."

Pregunta 5: ¿En qué año nació San Martín?
- "San Martín nació en 1816."



Al subir aHugging Face esta propuesta, y tras muchas batallas por hacer funcionar la interfaz, cambio de Llama a Qwen2.5-7B mediante API. No sólo funciona, sino que las respuestas son mejores a las obtenidas anteriormente.